In [2]:
# # Install Pytorch & other libraries
# %pip install "torch==2.4.1" tensorboard 
# %pip install flash-attn "setuptools<71.0.0" scikit-learn 
 
# # Install Hugging Face libraries
# %pip install  --upgrade \
#   "datasets==3.1.0" \
#   "accelerate==1.2.1" \
#   "hf-transfer==0.1.8"
#   #"transformers==4.47.1" \
 
# # ModernBERT is not yet available in an official release, so we need to install it from github
# %pip install "git+https://github.com/huggingface/transformers.git@6e0515e99c39444caae39472ee1b2fd76ece32f1" --upgrade

In [3]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [4]:
from datasets import load_dataset, concatenate_datasets
 
# Dataset id from huggingface.co/dataset
dataset_id = "youralien/feedback_qesconv_16wayclassification"
dataset_id_care = "youralien/CARE_10percent_16wayclassification"

# Load raw dataset
raw_dataset = load_dataset(dataset_id, split="train") # happens to be called train
care_raw_dataset = load_dataset(dataset_id_care, split="train") # happens to be called train

print(f"FeedbackESConv Raw dataset size: {len(raw_dataset)}")
print(f"CARE raw dataset size: {len(care_raw_dataset)}")

FeedbackESConv Raw dataset size: 8179
CARE raw dataset size: 370


In [5]:
split_dataset = raw_dataset.train_test_split(test_size=0.05, seed=0)
print(f"Train dataset size: {len(split_dataset['train'])}")
print(f"Test dataset size: {len(split_dataset['test'])}")
split_dataset['train'][0]

Train dataset size: 7770
Test dataset size: 409


{'conv_index': 252,
 'helper_index': 9,
 'input': ["Seeker: I am sorry. I can't chat any more please end of this survey. Ok. I can try.",
  'Helper: I am sorry to hear that. I believe in you, things will be better with time. Would you like to continue chatting?',
  'Seeker: Yes',
  'Helper: Okay. Are you excited for the upcoming holidays?',
  'Seeker: Yeah, i am excited upcoming chrisms and new year party.',
  'Helper: That sounds fun! Is the new year party with friends? Or is it a family affair?',
  "Seeker: New year party is with my friend and family affairs. It's very excited and lot of fun and games.",
  'Helper: That is fun! Friends and family are the most wonderful cure for the blues. What types of games will you be playing?'],
 'Reflections-goodareas': 0,
 'Validation-goodareas': 0,
 'Empathy-goodareas': 1,
 'Questions-goodareas': 1,
 'Suggestions-goodareas': 0,
 'Self-disclosure-goodareas': 0,
 'Structure-goodareas': 0,
 'Professionalism-goodareas': 0,
 'Reflections-badareas': 

In [6]:
eval_set = concatenate_datasets([split_dataset['test'], care_raw_dataset])
eval_set

Dataset({
    features: ['conv_index', 'helper_index', 'input', 'Reflections-goodareas', 'Validation-goodareas', 'Empathy-goodareas', 'Questions-goodareas', 'Suggestions-goodareas', 'Self-disclosure-goodareas', 'Structure-goodareas', 'Professionalism-goodareas', 'Reflections-badareas', 'Validation-badareas', 'Empathy-badareas', 'Questions-badareas', 'Suggestions-badareas', 'Self-disclosure-badareas', 'Structure-badareas', 'Professionalism-badareas', 'therapist_id', 'chat_code', 'therapist_index', 'Session Management-goodareas', 'Session Management-badareas'],
    num_rows: 779
})

In [7]:
split_dataset['test'] = eval_set

In [8]:
print(f"Train dataset size: {len(split_dataset['train'])}")
print(f"Test dataset size: {len(split_dataset['test'])}")

Train dataset size: 7770
Test dataset size: 779


### Targeted Sweep of Top Performing RoBERTa Hyperparams with Downsampling + Upweighting Majority Class

In [9]:
import torch
import gc

import evaluate
import numpy as np

def compute_metrics_fn(eval_preds):
    metrics = dict()
    
    accuracy_metric = evaluate.load('accuracy')
    precision_metric = evaluate.load('precision')
    recall_metric = evaluate.load('recall')
    f1_metric = evaluate.load('f1')
    
    logits = eval_preds.predictions
    labels = eval_preds.label_ids
    preds = np.argmax(logits, axis=-1)  
    
    metrics.update(accuracy_metric.compute(predictions=preds, references=labels))
    metrics.update(precision_metric.compute(predictions=preds, references=labels, average='binary'))
    metrics.update(recall_metric.compute(predictions=preds, references=labels, average='binary'))
    metrics.update(f1_metric.compute(predictions=preds, references=labels, average='binary'))

    # Print some predictions
    print(f"Some predictions: {preds[:10]}")
    
    return metrics
    
def cleanup(things_to_delete: list | None = None):
    if things_to_delete is not None:
        for thing in things_to_delete:
            if thing is not None:
                del thing

    gc.collect()
    torch.cuda.empty_cache()


In [10]:
# !pip install autoawq

In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer

# model_name = "Qwen/QwQ-32B-AWQ"

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype="float16", # auto
#     device_map="auto"
# )
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# prompt = "How many r's are in the word \"strawberry\""
# messages = [
#     {"role": "user", "content": prompt}
# ]
# text = tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True
# )

# model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# generated_ids = model.generate(
#     **model_inputs,
#     max_new_tokens=32768
# )
# generated_ids = [
#     output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
# ]

# response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
# print(response)


## Run a model on the validation dataset

In [23]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import json
import time
import logging
import os
from datetime import datetime

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("qwen_evaluation.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

class QwenCounselingEvaluator:
    def __init__(
        self, 
        model_name="Qwen/QwQ-32B-AWQ", 
        dtype="float16",
        max_tokens=128,
        temperature=0.1,
        output_dir="qwen_evaluation_results"
    ):
        """
        Initialize the evaluator with a Qwen model.
        
        Args:
            model_name (str): Hugging Face model ID for Qwen
            dtype (str): Model precision type
            max_tokens (int): Maximum tokens to generate for each prediction
            temperature (float): Sampling temperature
            output_dir (str): Directory to save evaluation results
        """
        self.model_name = model_name
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.output_dir = output_dir
        
        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        logger.info(f"Initializing QwenCounselingEvaluator with model: {model_name}")
        
        # Load model and tokenizer
        self.dtype = torch.float16 if dtype == "float16" else torch.bfloat16
        try:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=self.dtype,
                device_map="auto"
            )
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            logger.info(f"Successfully loaded model and tokenizer")
        except Exception as e:
            logger.error(f"Failed to load model: {e}")
            raise
    
    def evaluate_dataset(
        self, 
        dataset, 
        skill_class,
        context_size=5,
        save_predictions=True,
        save_responses=True
    ):
        """
        Evaluate the model on a dataset for a specific counseling skill.
        
        Args:
            dataset: HuggingFace dataset with train/test splits
            skill_class (str): The skill class to evaluate (e.g., "Reflections-goodareas")
            context_size (int): Number of prior messages to include as context
            save_predictions (bool): Whether to save predictions to disk
            save_responses (bool): Whether to save full model responses
            
        Returns:
            dict: Evaluation metrics
        """
        logger.info(f"Starting evaluation for skill: {skill_class}")
        
        # Prepare the dataset
        test_dataset = self._prepare_dataset(dataset, skill_class, context_size)
        
        # Extract labels
        if skill_class in test_dataset.features:
            labels = test_dataset[skill_class]
        else:
            # If the column has been renamed to "labels"
            labels = test_dataset["labels"] if "labels" in test_dataset.features else []
        
        start_time = time.time()
        
        # Process examples in batches
        self.all_predictions = []
        self.all_confidences = []
        self.all_responses = []

        num_errors = 0
        
        for i in tqdm(range(0, len(test_dataset)), desc=f"Evaluating {skill_class}"):

            example = test_dataset[i]

            example = self._prepare_input_text(example, context_size)
            
            # Create prompt for reflection prediction
            prompt = self._create_skill_prompt(example['text'], example['context'], skill_class)
            
            # Generate prediction
            response = self._generate_prediction(prompt)
            
            # Parse prediction
            prediction, confidence = self._parse_prediction(response, skill_class)
                    
            self.all_predictions.append(prediction)
            self.all_confidences.append(confidence)
            self.all_responses.append(response)
        
        elapsed_time = time.time() - start_time
        logger.info(f"Evaluation completed in {elapsed_time:.2f} seconds")
        
        # Compute metrics
        self.metrics = self._compute_metrics(self.all_predictions, labels, self.all_confidences)
        logger.info(f"Evaluation metrics: {metrics}")

        try:
            # Save results if requested
            if save_predictions or save_responses:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                
                results_data = {
                    "example_id": list(range(len(test_dataset))),
                    # "text": [ex["text"] for ex in test_dataset],
                    "true_label": labels if len(labels) > 0 else [None] * len(test_dataset),
                    "prediction": self.all_predictions,
                    "confidence": self.all_confidences
                }
                
                if save_responses:
                    results_data["model_response"] = self.all_responses
                
                # Create DataFrame and save to CSV
                results_df = pd.DataFrame(results_data)
                output_file = f"{self.output_dir}/{skill_class}_{timestamp}.csv"
                results_df.to_csv(output_file, index=False)
                logger.info(f"Saved results to {output_file}")
                
                # Save metrics as JSON
                metrics_file = f"{self.output_dir}/{skill_class}_metrics_{timestamp}.json"
                with open(metrics_file, 'w') as f:
                    json.dump(self.metrics, f, indent=2)
            
            return self.metrics
        except:
            return self.metrics
    
    def _prepare_dataset(self, dataset, skill_class, context_size):
        """
        Prepare dataset for evaluation.
        
        Args:
            dataset: HuggingFace dataset
            skill_class (str): The skill class to evaluate
            context_size (int): Number of prior messages to include as context
            
        Returns:
            dataset: Processed test dataset
        """
        logger.info("Preparing dataset for evaluation")
        
        # Use the test split
        test_dataset = dataset["test"]
        
        # Check if we need to rename the target column to "labels"
        if skill_class in test_dataset.features and "labels" not in test_dataset.features:
            test_dataset = test_dataset.rename_column(skill_class, "labels")
        
        return test_dataset

    def _prepare_input_text(self, example, context_size=1):
        """
        [-6] Seeker: 
        [-5] Helper:
        [-4] Seeker: 
        [-3] Helper:
        [-2] Seeker: 
        [-1] Helper: Response to classify
        """
        # Convert the last two items of input list to a single text
        response_to_classify = example['input'][-1]
        if context_size is None:
            context = "\n".join(example['input'][:-1])
        else:
            context_start_idx = -1 - context_size
            context = "\n".join(example['input'][context_start_idx:-1])
        return {
            'text': f"{response_to_classify}",
            'context': context,
            **{k:v for k,v in example.items() if k != 'input'}  # Keep other fields
        }
    
    def _create_skill_prompt(self, text, conversation_history=None, skill_class="Reflections"):
        """
        Create a prompt for the model to detect a counseling skill.
        
        Args:
            text (str): The counselor's response text
            conversation_history (str, optional): Previous messages
            skill_class (str): The skill to evaluate
            
        Returns:
            str: Formatted prompt
        """
        # Extract main skill from class name (e.g., "Reflections-goodareas" -> "Reflections")
        skill = skill_class.split('-')[0] if '-' in skill_class else skill_class
        
        # Define skill description
        skill_descriptions = {
            "Reflections": "restating or rephrasing what the client has expressed to show understanding",
            "Validation": "acknowledging and accepting the client's emotions or experiences as valid",
            "Empathy": "demonstrating understanding of the client's emotional experience",
            "Questions": "asking the client to provide information or explore thoughts and feelings",
            "Suggestions": "offering ideas, advice, or potential solutions to the client",
            "Self-disclosure": "sharing personal information or experiences with the client",
            "Structure": "guiding the session or conversation in a purposeful way",
            "Professionalism": "maintaining ethical boundaries and professional standards"
        }
        
        skill_description = skill_descriptions.get(
            skill, 
            "an important counseling technique that helps clients feel understood"
        )
        
        context_text = f"\nConversation History: \"{conversation_history}\"\n" if conversation_history else ""
        
        prompt = f"""You are an expert in analyzing counseling techniques. Determine if the following counselor response contains the skill of {skill}.

{skill} involves {skill_description}.

{context_text}
Counselor's response: "{text}"

Analysis instructions:
1. Identify any language in the response that might represent the {skill} technique.
2. Consider whether the counselor is explicitly using this technique or not.
3. Conclude with a clear YES or NO judgment.
4. After your internal thinking, provide ONLY a single-word answer: YES or NO. 

FINAL ANSWER: (YES/NO)"""

        return prompt
    
    def _generate_prediction(self, prompt):
        """
        Generate a prediction from the model.
        
        Args:
            prompt (str): The formatted prompt
            
        Returns:
            str: Model's response
        """
        # Format as chat
        messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        # Tokenize
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        
        # Generate
        with torch.no_grad():
            generated_ids = self.model.generate(
                **model_inputs,
                max_new_tokens=self.max_tokens,
                temperature=self.temperature,
                do_sample=False  # Deterministic for evaluation
            )
        
        # Extract only the new tokens
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        # Decode
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        return response
    
    def _parse_prediction(self, response, skill_class):
        """
        Parse the model's response to extract the prediction.
        
        Args:
            response (str): The model's response
            skill_class (str): The skill being evaluated
            
        Returns:
            tuple: (prediction, confidence)
                - prediction (int): 1 if skill present, 0 if not
                - confidence (float): Confidence score (0.5-1.0)
        """
        response_lower = response.lower()
        
        # Check for explicit final answer
        if "final answer: yes" in response_lower or "final answer: (yes)" in response_lower:
            return 1, 0.9
        elif "final answer: no" in response_lower or "final answer: (no)" in response_lower:
            return 0, 0.9
            
        # Look for yes/no at the end
        lines = response_lower.split('\n')
        for line in reversed(lines):
            if "yes" in line and "no" not in line:
                return 1, 0.8
            elif "no" in line and "yes" not in line:
                return 0, 0.8
        
        # Count yes/no mentions
        yes_count = response_lower.count(" yes ")
        no_count = response_lower.count(" no ")
        
        if yes_count > no_count:
            confidence = 0.5 + min((yes_count - no_count) * 0.1, 0.3)
            return 1, confidence
        elif no_count > yes_count:
            confidence = 0.5 + min((no_count - yes_count) * 0.1, 0.3)
            return 0, confidence
        else:
            # Default to no with low confidence if unclear
            return 0, 0.5
    
    def _compute_metrics(self, predictions, labels, confidences=None):
        """
        Compute evaluation metrics.
        
        Args:
            predictions (list): Model predictions
            labels (list): Ground truth labels
            confidences (list, optional): Confidence scores for predictions
            
        Returns:
            dict: Evaluation metrics
        """
        if not labels or len(labels) == 0:
            logger.warning("No labels provided for evaluation")
            return {"error": "No labels available for evaluation"}
        
        # Convert to numpy arrays
        y_pred = np.array(predictions)
        y_true = np.array(labels)
        
        # Basic classification report
        report = classification_report(y_true, y_pred, output_dict=True)
        
        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        
        metrics = {
            "accuracy": report["accuracy"],
            "precision": report["1"]["precision"] if 1 in report else 0,
            "recall": report["1"]["recall"] if 1 in report else 0,
            "f1-score": report["1"]["f1-score"] if 1 in report else 0,
            "support": int(report["1"]["support"]) if 1 in report else 0,
            "confusion_matrix": {
                "true_negative": int(cm[0, 0]),
                "false_positive": int(cm[0, 1]),
                "false_negative": int(cm[1, 0]),
                "true_positive": int(cm[1, 1])
            }
        }
        
        # Add confidence metrics if provided
        if confidences:
            conf_array = np.array(confidences)
            metrics["mean_confidence"] = float(np.mean(conf_array))
            metrics["confidence_correct"] = float(np.mean(conf_array[y_pred == y_true]))
            metrics["confidence_incorrect"] = float(np.mean(conf_array[y_pred != y_true])) if np.any(y_pred != y_true) else 0
        
        return metrics
    
    def cleanup(self):
        """Release resources."""
        del self.model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        logger.info("Resources cleaned up")

In [17]:
split_dataset['test'].features

{'conv_index': Value(dtype='int64', id=None),
 'helper_index': Value(dtype='int64', id=None),
 'input': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
 'Reflections-goodareas': Value(dtype='int64', id=None),
 'Validation-goodareas': Value(dtype='int64', id=None),
 'Empathy-goodareas': Value(dtype='int64', id=None),
 'Questions-goodareas': Value(dtype='int64', id=None),
 'Suggestions-goodareas': Value(dtype='int64', id=None),
 'Self-disclosure-goodareas': Value(dtype='int64', id=None),
 'Structure-goodareas': Value(dtype='int64', id=None),
 'Professionalism-goodareas': Value(dtype='int64', id=None),
 'Reflections-badareas': Value(dtype='int64', id=None),
 'Validation-badareas': Value(dtype='int64', id=None),
 'Empathy-badareas': Value(dtype='int64', id=None),
 'Questions-badareas': Value(dtype='int64', id=None),
 'Suggestions-badareas': Value(dtype='int64', id=None),
 'Self-disclosure-badareas': Value(dtype='int64', id=None),
 'Structure-badareas': Value(dtype='in

In [25]:
qwen_evaluator = QwenCounselingEvaluator(max_tokens=8192, output_dir="qwen_eval_results")

Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:02<00:00,  1.73it/s]


In [24]:
qwen_evaluator.cleanup()

### Testing the code on one case, to develop intuition about internals

In [28]:
# Prepare the dataset
test_dataset = qwen_evaluator._prepare_dataset(mini_split_dataset, skill_class="Reflections-goodareas", context_size=5)

# Extract labels
if "Reflections-goodareas" in test_dataset.features:
    labels = test_dataset[skill_class]
else:
    # If the column has been renamed to "labels"
    labels = test_dataset["labels"] if "labels" in test_dataset.features else []

In [59]:
# Get client context if available
example = test_dataset[0]
prepped_example = prepare_input_text(example)

prompt = qwen_evaluator._create_skill_prompt(prepped_example['text'], prepped_example['context'], "Reflections-goodareas")
print(prompt)

You are an expert in analyzing counseling techniques. Determine if the following counselor response contains the skill of Reflections.

Reflections involves restating or rephrasing what the client has expressed to show understanding.


Conversation History: "Seeker: Well, my manager has decided that everyone in the office will continue to work from after the pandemic is under control. But I've found myself depressed from working from home all the time."

Counselor's response: "Helper: I can hear that this shift to working from home has been really challenging for you and it's leading to feelings of depression. Can you tell me more about your experience and how it has been affecting you?"

Analysis instructions:
1. Identify any language in the response that might represent the Reflections technique.
2. Consider whether the counselor is explicitly using this technique or not.
3. Conclude with a clear YES or NO judgment.
4. After your internal thinking, provide ONLY a single-word answer: 

In [60]:
# Generate prediction
response = qwen_evaluator._generate_prediction(prompt)

In [54]:
print(response)

Okay, let's tackle this. The user wants to know if the counselor's response uses the Reflections technique. First, I need to recall what Reflections mean in counseling. From what I remember, Reflections involve restating or rephrasing the client's words to show understanding. It's about reflecting back the content and possibly the feelings the client expressed.

Looking at the conversation: The seeker says their manager wants everyone to keep working from home even after the pandemic, and they're feeling depressed because of it. The counselor responds by saying, "I can hear that this shift to working from home has been really challenging for you and it's leading to feelings of depression. Can you tell me more about your experience and how it has been affecting you?"

Now, breaking down the counselor's response. The first part: "I can hear that this shift to working from home has been really challenging for you and it's leading to feelings of depression." Here, the counselor is rephrasi

In [61]:
# New prompt that is condensed.
print(response)

Okay, let's tackle this. The user wants to know if the counselor's response uses the Reflections technique. First, I need to recall what Reflections entail. The definition given is restating or rephrasing the client's words to show understanding. 

Looking at the conversation: The seeker says their manager wants everyone to keep working from home even after the pandemic, and they're feeling depressed because of it. 

The counselor responds by saying, "I can hear that this shift to working from home has been really challenging for you and it's leading to feelings of depression. Can you tell me more about your experience and how it has been affecting you?"

Breaking it down, the counselor starts by acknowledging the client's situation. The phrase "this shift to working from home has been really challenging for you and it's leading to feelings of depression" seems to restate the client's points. The client mentioned being depressed from working from home, and the counselor reflects that b

In [62]:
# Parse prediction
prediction, confidence = qwen_evaluator._parse_prediction(response, "Reflections-goodareas")
print(prediction)
print(confidence)

1
0.8


### Run sweep on the evaluation dataset

In [26]:
eval_metrics = qwen_evaluator.evaluate_dataset(dataset=split_dataset, skill_class="Reflections-goodareas")

Evaluating Reflections-goodareas:   0%|                                                                                                                 | 0/779 [00:00<?, ?it/s]/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`.

NameError: name 'metrics' is not defined

In [29]:
qwen_evaluator.metrics

{'accuracy': 0.7060333761232349,
 'precision': 0,
 'recall': 0,
 'f1-score': 0,
 'support': 0,
 'confusion_matrix': {'true_negative': 470,
  'false_positive': 222,
  'false_negative': 7,
  'true_positive': 80},
 'mean_confidence': 0.8002567394094994,
 'confidence_correct': 0.7999999999999998,
 'confidence_incorrect': 0.800873362445415}

'Okay, let\'s tackle this. The user wants to know if the counselor\'s response uses the Reflections technique. First, I need to recall what Reflections means. It\'s when the counselor restates or rephrases the client\'s words to show understanding. \n\nLooking at the conversation history, the seeker is talking about a breakup and feeling confused. The counselor\'s response in question is: "I understand this is really hard for you. What do you think could be the next step for you?"\n\nBreaking it down. The first part, "I understand this is really hard for you," seems like it\'s reflecting the seeker\'s emotion. The seeker mentioned being in an argument and loving the person even after the breakup. The counselor acknowledges the difficulty, which might be a reflection of the client\'s feelings. \n\nBut wait, Reflections usually rephrase what the client said, not just acknowledge emotions. The seeker\'s last message was "I have no idea, what to do now." The counselor\'s second sentence as

In [44]:
SKILL_CLASS= "Reflections-goodareas"
test_dataset = qwen_evaluator._prepare_dataset(split_dataset, "Reflections-goodareas", context_size=5)
        
# Extract labels
if SKILL_CLASS in test_dataset.features:
    labels = test_dataset[SKILL_CLASS]
else:
    # If the column has been renamed to "labels"
    labels = test_dataset["labels"] if "labels" in test_dataset.features else []

In [46]:
y_pred = np.array(qwen_evaluator.all_predictions)
y_true = np.array(labels)

# Basic classification report
report = classification_report(y_true, y_pred, output_dict=True)

In [47]:
report

{'0': {'precision': 0.9853249475890985,
  'recall': 0.6791907514450867,
  'f1-score': 0.8041060735671514,
  'support': 692.0},
 '1': {'precision': 0.26490066225165565,
  'recall': 0.9195402298850575,
  'f1-score': 0.41131105398457585,
  'support': 87.0},
 'accuracy': 0.7060333761232349,
 'macro avg': {'precision': 0.6251128049203771,
  'recall': 0.799365490665072,
  'f1-score': 0.6077085637758636,
  'support': 779.0},
 'weighted avg': {'precision': 0.904866779650257,
  'recall': 0.7060333761232349,
  'f1-score': 0.7602380803660165,
  'support': 779.0}}

In [49]:
test_dataset

Dataset({
    features: ['conv_index', 'helper_index', 'input', 'labels', 'Validation-goodareas', 'Empathy-goodareas', 'Questions-goodareas', 'Suggestions-goodareas', 'Self-disclosure-goodareas', 'Structure-goodareas', 'Professionalism-goodareas', 'Reflections-badareas', 'Validation-badareas', 'Empathy-badareas', 'Questions-badareas', 'Suggestions-badareas', 'Self-disclosure-badareas', 'Structure-badareas', 'Professionalism-badareas', 'therapist_id', 'chat_code', 'therapist_index', 'Session Management-goodareas', 'Session Management-badareas'],
    num_rows: 779
})

In [51]:
results_data = {
    "example_id": list(range(len(test_dataset))),
    "input": [ex["input"] for ex in test_dataset],
    "true_label": labels if len(labels) > 0 else [None] * len(test_dataset),
    "prediction": qwen_evaluator.all_predictions,
    "confidence": qwen_evaluator.all_confidences
}


#labels = test_dataset[skill_class]
results_data["model_response"] = qwen_evaluator.all_responses
                
# Create DataFrame and save to CSV
results_df = pd.DataFrame(results_data)
# output_file = f"{qwen_evaluator.output_dir}/{SKILL_CLASS}.csv"
# results_df.to_csv(output_file, index=False)
output_file = f"{qwen_evaluator.output_dir}/{SKILL_CLASS}.json"
results_df.to_json(output_file, index=False)

In [52]:
import json

# Save directly to JSON
output_file = f"{qwen_evaluator.output_dir}/{SKILL_CLASS}-clean.json"
with open(output_file, 'w') as f:
    json.dump(results_data, f, indent=4)

In [53]:
import json

# Get true labels or use None if not available
true_labels = labels if len(labels) > 0 else [None] * len(test_dataset)

# Create zipped data with list comprehension
zipped_results = [
    {
        "example_id": i,
        "input": inp,
        "true_label": label,
        "prediction": pred,
        "confidence": conf,
        "model_response": resp
    }
    for i, inp, label, pred, conf, resp in zip(
        range(len(test_dataset)),
        [ex["input"] for ex in test_dataset],
        true_labels,
        qwen_evaluator.all_predictions,
        qwen_evaluator.all_confidences,
        qwen_evaluator.all_responses
    )
]

# Save to JSON
output_file = f"{qwen_evaluator.output_dir}/{SKILL_CLASS}-zipped.json"
with open(output_file, 'w') as f:
    json.dump(zipped_results, f, indent=4)

## Making predictions on Test/Study Data

In [41]:
import pandas as pd

# condition = "control"
# condition = "treatment"
# input_data = pd.read_csv(f"./all_{condition}_seekerhelper_pairs.csv")
input_data = pd.read_csv("N94_all_seekerhelper_pairs.csv")
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c..."
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...


In [42]:
from datasets import Dataset

study_dataset = Dataset.from_pandas(input_data)

wandb: ERROR Problem finishing run


In [43]:
from transformers import pipeline

# WHICH_CLASS="Reflections-goodareas"
WHICH_CLASS="Questions-goodareas"
# load model from huggingface.co/models using our repository id
# classifier = pipeline("sentiment-analysis", model=f"./roberta-Reflections-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-d6x1jzik-1741277930", device=0)
classifier = pipeline("sentiment-analysis", model=f"./roberta-Questions-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-82jc07j0-1741329550", device=0)

# classifier = pipeline("sentiment-analysis", model="ModernBERT-Empathy-goodareas-classifier", device=0)

# sample = f"Seeker: {input_data.loc[0, "seeker_post"]}\nHelper: {input_data.loc[0, "response_post"]}"
# pred = classifier(sample)
# print(pred)

def binary_prediction_seeker_response_post(seeker, helper):
    # sample = f"Seeker: {seeker}\nHelper: {helper}"
    sample = f"Seeker: {seeker}[SEP]Helper: {helper}"
    
    pred = classifier(sample)
    return int(pred[0]['label'] == 'selected')

Device set to use cuda:0


In [44]:
# output_preds = input_data.apply(binary_prediction_seeker_response_post, axis=0)

def predict_reflection(example):
    # Apply your binary prediction function to each example
    example["prediction"] = binary_prediction_seeker_response_post(
        example["seeker_post"], 
        example["response_post"]
    )
    return example

# Apply the function to the entire dataset at once
predicted_dataset = study_dataset.map(predict_reflection)

# strengths = [binary_prediction_seeker_response_post(input_data.loc[i, "seeker_post"], input_data.loc[i, "response_post"])
#              for i in range(len(input_data))]

Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3842/3842 [00:38<00:00, 99.94 examples/s]


In [45]:
input_data[f"{WHICH_CLASS}"] = predicted_dataset['prediction']
# input_data[f"{WHICH_CLASS}"] = strengths
# input_data["Empathy-goodareas"] = strengths

In [46]:
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post,Questions-goodareas
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...,0
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...,0
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...,0
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c...",0
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...,1


In [47]:
# input_data.to_csv(f'all_{condition}_seekerhelper_pairs_{WHICH_CLASS}.csv')
input_data.to_csv(f"N94_all_seekerhelper_pairs_{WHICH_CLASS}.csv")

In [37]:
f'all_{condition}_seekerhelper_pairs_{WHICH_CLASS}.csv'

'all_control_seekerhelper_pairs_Reflections-goodareas.csv'